**<h1 align="center"> Исследование алгоритма k-ближайших соседей (KNN) на примере классификации ирисов (Iris Species)**

---
<h1 align="center">1.Первичный анализ и подготовка данных


In [ ]:
import kagglehub
import pandas as pd
import os

# Download latest version
path = kagglehub.dataset_download("yasserh/wine-quality-dataset")

full_path = os.path.join(path, 'WineQT.csv')

df = pd.read_csv(full_path)
print(df.head())
print(df.info())

Using Colab cache for faster access to the 'wine-quality-dataset' dataset.
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol 

<h2 style="color: #722f37; border-bottom: 2px solid #722f37; padding-bottom: 10px;">🍷 Анализ качества вина (Wine Quality)</h2>

<p><i>Решила взять этот датасет, потому что на Ирисах всё было слишком идеально. Тут данные «жизненные»:</i></p>

<ul style="line-height: 1.6;">
    <li><b>Размер:</b> 1143 записи.</li>
    <li><b>Состав:</b> 11 химических параметров (кислотность, сахар, алкоголь и др.).</li>
    <li><b>Цель:</b> Предсказать экспертную оценку качества (от 3 до 8).</li>
    <li><b>Нюанс:</b> Пропусков нет, но разброс цифр огромный (от 0.1 до 289). Без масштабирования KNN просто «ослепнет».</li>
</ul>

<h3 style="color: #2e7d32; margin-top: 20px;">🛠️ Подготовка (коротко о главном)</h3>

<div style="background-color: #ffffff; padding: 15px; border-left: 5px solid #2e7d32; border-radius: 5px;">
    <ol style="margin: 0; padding-left: 20px; line-height: 1.6;">
        <li><b>Удаление Id:</b> Технический столбец, который только путает модель.</li>
        <li><b>Split 80/20:</b> Классика — учимся на большинстве, проверяем на остатке.</li>
        <li><b>StandardScaler:</b> Приводим всё к единому масштабу. KNN считает расстояния, и без этого признаки с большими числами (как диоксид серы) «задавили» бы остальные.</li>
        <li><b>Важное правило:</b> Обучаем скалер только на <code>train</code>, чтобы избежать «утечки данных» из будущего.</li>
    </ol>
</div>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(['Id', 'quality'], axis=1)
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"У нас есть {len(X_train)} бокалов вина для обучения и {len(X_test)} для экзамена.")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Данные подготовлены и отмасштабированы.")

У нас есть 914 бокалов вина для обучения и 229 для экзамена.
Данные подготовлены и отмасштабированы.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Обучаем на сырых данных
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
acc_raw = accuracy_score(y_test, knn_raw.predict(X_test))

# Обучаем на масштабированных данных
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
acc_scaled = accuracy_score(y_test, knn_scaled.predict(X_test_scaled))

print(f"Точность БЕЗ масштабирования: {acc_raw:.4f}")
print(f"Точность С масштабированием: {acc_scaled:.4f}")

Точность БЕЗ масштабирования: 0.5153
Точность С масштабированием: 0.5590


### 📊 Сравнение результатов (Baseline vs Scaled)

На этом этапе мы проверили работу KNN "из коробки" и увидели реальную разницу:

| Метод подготовки | Точность (Accuracy) |
| :--- | :--- |
| **Без масштабирования** | <span style="color: #d32f2f;">0.5153</span> |
| **С StandardScaler** | <span style="color: #2e7d32;">**0.5590**</span> |

> **Вывод:** Масштабирование дало прирост точности более чем на **4%**. Это подтверждает, что в данных о вине признаки имеют слишком разный масштаб.